In [ ]:
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

mlflow.set_tracking_uri('http://18.219.3.159:5000')
client = mlflow.tracking.MlflowClient()

In [ ]:
# Load all runs from the experiment
experiment = client.get_experiment_by_name('melanoma-detection')
if experiment is None:
    print('No experiment found. Run scripts/train.py first.')
else:
    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=['metrics.val/auroc DESC']
    )
    runs_df = pd.DataFrame([
        {
            'run_id': r.info.run_id[:8],
            'backbone': r.data.params.get('model.backbone', '?'),
            'lr': r.data.params.get('training.lr', '?'),
            'val_auroc': r.data.metrics.get('val/auroc', np.nan),
            'val_f1': r.data.metrics.get('val/f1', np.nan),
        }
        for r in runs
    ])
    print(runs_df.to_string(index=False))

In [ ]:
# Learning curves for the best run
if experiment is not None and runs:
    best_run = runs[0]
    
    def get_metric_history(run_id, metric):
        history = client.get_metric_history(run_id, metric)
        return [(h.step, h.value) for h in history]
    
    train_loss = get_metric_history(best_run.info.run_id, 'train/loss_epoch')
    val_loss   = get_metric_history(best_run.info.run_id, 'val/loss')
    val_auroc  = get_metric_history(best_run.info.run_id, 'val/auroc')
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    
    if train_loss:
        epochs, vals = zip(*train_loss)
        axes[0].plot(epochs, vals, label='Train loss', color='steelblue')
    if val_loss:
        epochs, vals = zip(*val_loss)
        axes[0].plot(epochs, vals, label='Val loss', color='tomato', linestyle='--')
    axes[0].set_title('Focal Loss — Train vs Val')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    
    if val_auroc:
        epochs, vals = zip(*val_auroc)
        axes[1].plot(epochs, vals, color='mediumseagreen', marker='o', markersize=4)
        axes[1].axhline(y=max(vals), color='gray', linestyle=':', alpha=0.7)
        axes[1].set_title(f'Val AUC-ROC (best = {max(vals):.4f})')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('AUC-ROC')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Clinical metrics at calibrated threshold
from cancer_detection.evaluation.metrics import compute_metrics
import json, torch

threshold_path = '../artifacts/threshold.json'
try:
    threshold = json.load(open(threshold_path))['threshold']
    print(f'Calibrated threshold: {threshold:.4f}')
    print('Load test set predictions and call compute_metrics(y_true, y_prob, threshold)')
    print('Metrics include: auroc, f1, sensitivity, specificity, pauc')
except FileNotFoundError:
    print('Run training first to generate artifacts/threshold.json')